In [38]:
from ESRNN.utils_evaluation import evaluate_prediction_owa
from ESRNN import ESRNN
from ESRNN.utils_evaluation import Naive2, Naive


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df = pd.read_csv(
    "/home/erti/PROJECT_repos/ForeST_HPO/hpo-innosuisse/data/2024-05-27 DE Auftragseingang Maschinenbau 28.00.csv"
)
df.columns

Index(['Datum', 'Insgesamt', 'Inland', 'Ausland', 'Ausland (Eurozone)',
       'Ausland (Nicht-Eurozone)'],
      dtype='object')

In [18]:
y_df = pd.DataFrame()
y_df["ds"] = pd.to_datetime(df["Datum"])
y_df["y"] = df["Insgesamt"]
y_df["unique_id"] = "Insgesamt"


In [20]:
assert y_df.isna().sum().sum() == 0, "There are NaN values in the data"

In [ ]:
x_df = y_df.drop("y", 1)
x_df["x"] = "Macro"  # TODO

In [ ]:
SPLIT_IDX = y_df[y_df.ds == pd.to_datetime("2017-01-31")].index[0]
print(SPLIT_IDX, SPLIT_IDX / len(y_df))

y_train_df = y_df.iloc[:SPLIT_IDX]
y_test_df = y_df.iloc[SPLIT_IDX:]

X_train_df = x_df.iloc[:SPLIT_IDX]
X_test_df = x_df.iloc[SPLIT_IDX:]

312 0.7819548872180451


In [44]:
# naive = Naive() # TODO use Naive2 or SesonalNaive
# naive.fit(y_test_df)

y_test_df["y_hat_naive2"] = y_test_df["y"].shift(1).fillna(method="bfill")

/tmp/nix-shell-268640-0/ipykernel_277122/1444109779.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test_df["y_hat_naive2"] = y_test_df["y"].shift(1).fillna(method="bfill")


In [47]:
# Instantiate model
model = ESRNN(
    max_epochs=25,
    freq_of_test=5,
    batch_size=1, # assert self.batch_size <= self.n_series
    learning_rate=1e-4,
    per_series_lr_multip=0.8,
    lr_scheduler_step_size=10,
    lr_decay=0.1,
    gradient_clipping_threshold=50,
    rnn_weight_decay=0.0,
    level_variability_penalty=100,
    testing_percentile=50,
    training_percentile=50,
    ensemble=False,
    max_periods=25,
    seasonality=[],
    input_size=4,
    output_size=6,
    cell_type="LSTM",
    state_hsize=40,
    dilations=[[1], [6]],
    add_nl_layer=False,
    random_seed=1,
    device="cuda",  # FIXED ERTI
)

In [ ]:
# import torch
# torch.cuda.set_device(0)
# torch.set_default_device("cuda") # FIXED ERTI (maybe not needed with the above device)

In [48]:
# Fit model
# If y_test_df is provided the model
# will evaluate predictions on
# this set every freq_test epochs
model.fit(
    X_train_df,
    y_train_df,
    X_test_df,
    y_test_df,
)


Infered frequency: M
=============== Training ESRNN  ===============

========= Epoch 0 finished =========
Training time: 0.10705
Training loss (50 prc): 0.79760
Testing loss  (50 prc): 0.05437
OWA: nan 
SMAPE: nan 
MASE: nan 
========= Epoch 1 finished =========
Training time: 0.08611
Training loss (50 prc): 0.79712
========= Epoch 2 finished =========
Training time: 0.08785
Training loss (50 prc): 0.79665
========= Epoch 3 finished =========
Training time: 0.07919
Training loss (50 prc): 0.79618
========= Epoch 4 finished =========
Training time: 0.08459
Training loss (50 prc): 0.79570
========= Epoch 5 finished =========
Training time: 0.08623
Training loss (50 prc): 0.79524
Testing loss  (50 prc): 0.05232
OWA: nan 
SMAPE: nan 
MASE: nan 
========= Epoch 6 finished =========
Training time: 0.07705
Training loss (50 prc): 0.79477
========= Epoch 7 finished =========
Training time: 0.07764
Training loss (50 prc): 0.79430
========= Epoch 8 finished =========
Training time: 0.08035
Trai

In [49]:
import pickle

with open("forest_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [50]:
# Predict on test set
y_hat_df = model.predict(X_test_df)

# Evaluate predictions
final_owa, final_mase, final_smape = evaluate_prediction_owa(
    y_hat_df, y_train_df, X_test_df, y_test_df, naive2_seasonality=1
)

===============  Model evaluation  ==============
OWA: nan 
SMAPE: nan 
MASE: nan 


In [ ]:
# import matplotlib.pyplot as plt
# ImportError: Matplotlib requires numpy>=1.23; you have 1.16.6
# pip install matplotlib==3.2.2

ImportError: Matplotlib requires numpy>=1.23; you have 1.16.6

In [58]:
y_hat_df["y_test"] = y_test_df["y"]

In [59]:
y_hat_df.to_csv("y_hat_df.csv")